In [ ]:
# Example 1: Create OmegaConf from a Python dict + basic attribute access.
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "server": {"host": "localhost", "port": 8080},
    "debug": True,
    "workers": 4,
})

print(f"server.host = {cfg.server.host}")
print(f"server.port = {cfg.server.port}")
print(f"debug = {cfg.debug}")
print(f"workers = {cfg.workers}")

# Both attribute and dict-style access work.
assert cfg.server.host == "localhost"
assert cfg["server"]["port"] == 8080
assert cfg.workers == 4


In [ ]:
# Example 2: YAML round-trip (load YAML string -> OmegaConf -> back to YAML).
from omegaconf import OmegaConf

yaml_str = """
database:
  driver: postgres
  host: db.example.com
  port: 5432
features:
  - auth
  - billing
  - reporting
"""

cfg = OmegaConf.create(yaml_str)
print(f"driver = {cfg.database.driver}")
print(f"features = {list(cfg.features)}")

out = OmegaConf.to_yaml(cfg)
print("\nRendered YAML:")
print(out)

assert cfg.database.port == 5432
assert "auth" in cfg.features


In [ ]:
# Example 3: Merge multiple configs (base + override pattern).
from omegaconf import OmegaConf

base = OmegaConf.create({
    "model": {"name": "resnet50", "lr": 0.001, "batch_size": 32},
    "epochs": 10,
})

override = OmegaConf.create({
    "model": {"lr": 0.0001},
    "epochs": 50,
})

merged = OmegaConf.merge(base, override)
print(OmegaConf.to_yaml(merged))

# Override wins for overlapping keys; non-overlapping keys are preserved.
assert merged.model.lr == 0.0001
assert merged.model.name == "resnet50"
assert merged.model.batch_size == 32
assert merged.epochs == 50


In [ ]:
# Example 4: Variable interpolation (${...} references).
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "user": "alice",
    "root_dir": "/data",
    "user_dir": "${root_dir}/${user}",
    "backup_dir": "${user_dir}/backup",
})

print(f"user_dir = {cfg.user_dir}")
print(f"backup_dir = {cfg.backup_dir}")

assert cfg.user_dir == "/data/alice"
assert cfg.backup_dir == "/data/alice/backup"

# Interpolation is dynamic; change a root and downstream values update.
cfg.user = "bob"
assert cfg.user_dir == "/data/bob"
assert cfg.backup_dir == "/data/bob/backup"


In [ ]:
# Example 5: Structured config from a dataclass + type safety.
from dataclasses import dataclass, field
from typing import List

from omegaconf import OmegaConf


@dataclass
class TrainConfig:
    lr: float = 0.01
    epochs: int = 100
    layers: List[int] = field(default_factory=lambda: [64, 128, 256])


cfg = OmegaConf.structured(TrainConfig)
print(OmegaConf.to_yaml(cfg))

assert cfg.lr == 0.01
assert cfg.epochs == 100
assert list(cfg.layers) == [64, 128, 256]

# Typed merge with overrides + container conversion.
merged = OmegaConf.merge(cfg, OmegaConf.create({"lr": 0.05, "epochs": 200}))
container = OmegaConf.to_container(merged, resolve=True)
print(f"\nContainer: {container}")
assert container == {"lr": 0.05, "epochs": 200, "layers": [64, 128, 256]}
